# HDB Rentals 4: follow-up experiments

Experiments that test claims in `overall_idea.md` which the existing runs cannot settle. Each section states the
claim, the decision rule (fixed before running), and **where the result goes** once it has run.

| ID | Claim tested | `overall_idea.md` step | Status |
|---|---|---|---|
| X1 | Tree and neural premium models complement each other, on the same market level | 2, 3 | to run (long) |
| X2 | Cold-start blocks are too rare to move test RMSE | 2 | ran 2026-10-30 |

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]
import compare_gnn_nb10 as cmp  # noqa: E402  (data paths, cleaning, split dates; LightGBM is not used here)


def run_script(name, *args):
    """Run a GNN script in its own process (keeps torch and LightGBM out of this kernel)."""
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *args], cwd=ROOT, check=True)

## X1. Tree vs GNN premium on the same market level

**Why.** In Part 2.4 the GNN runs use the `damped` level while E1 LightGBM uses `MONTH_INDEX`. `damped` over-predicts
on the holdout (+26 bias, Part 2.3 E6), so the GNN's +19 vs LightGBM's -37 bias, and the GNN + E1 blend gain
(483.1 to 480.5), may come from the level, not from the premium models. The smoke run already shows E1 on the `blend`
level at bias +3 (holdout 481.65, matching E6's 481.7).

**Change.** `gnn_premiums_blend.py --level blend --same-level --premiums row`: both models learn rent / the `blend`
level (Part 2.3's kept level), so they differ only in the premium model. Seeds 0, 1, 2 for the neural models.

**Decision rules.**
- *Complementary* (step 3) if the tuning split picks a blend weight strictly between 0 and 1 **and** the blend
  beats E1 on the holdout by more than twice the neural models' seed standard deviation.
- *Opposite errors* stays in step 3 only if the holdout biases of E1 and the GNN differ in sign.
- *Graph adds no accuracy* (step 2) stands if GNN and MLP differ by less than twice their seed standard deviation.

In [ ]:
# Each seed trains MLP_matched and GNN_uniform on both splits; long on a laptop CPU.
RUN_X1 = False
SEEDS = [0, 1, 2]
if RUN_X1:
    for seed in SEEDS:
        run_script("gnn_premiums_blend.py", "--level", "blend", "--same-level", "--premiums", "row",
                   "--seed", str(seed))

In [ ]:
X1 = ROOT / "outputs/gnn_premiums_blend"
res = pd.concat([pd.read_csv(X1 / f"level-blend_same_seed{s}" / "results.csv") for s in SEEDS])
by_model = res.groupby(["split", "model"]).agg(rmse=("rmse", "mean"), rmse_sd=("rmse", "std"),
                                               bias=("bias", "mean"), weight=("weight", "mean"))
by_model.round(2)

By history: holdout rows bucketed by how many fit rentals their block + flat type has (as
`outputs/gnn_neighbour_premiums/by_history.csv`).

In [ ]:
raw = cmp.clean_rent(pd.read_csv(cmp.DATA / "train.csv"))
raw["row_id"] = np.arange(len(raw))  # the scripts' row_id: train.csv order after cleaning
d = raw["RENT_APPROVAL_DATE"]
keys = cmp.KEY + ["FLAT_TYPE"]
counts = raw[d < cmp.HOLD_START].groupby(keys).size().rename("n").reset_index()
hist = raw[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)].merge(counts, on=keys, how="left")[["row_id", "n"]].fillna(0)
hist["history"] = pd.cut(hist["n"], [-1, 0, 2, 5, 10, np.inf], labels=["0", "1-2", "3-5", "6-10", ">10"])

pred = pd.concat([pd.read_csv(X1 / f"level-blend_same_seed{s}" / "predictions.csv") for s in SEEDS])
pred = pred[pred.split == "holdout"].merge(hist, on="row_id", validate="m:1")
rows = []
for (model, history), g in pred.groupby(["model", "history"], observed=True):
    for name, col in [("LightGBM E1", "lightgbm"), (model, "neural")]:
        err = g[col] - g["MONTHLY_RENT"]
        rows.append(dict(model=name, history=history, n=len(g) // len(SEEDS),
                         rmse=np.sqrt((err ** 2).mean()), bias=err.mean()))
pd.DataFrame(rows).drop_duplicates(["model", "history"]).pivot(index="history", columns="model",
                                                               values=["rmse", "bias"]).round(1)

**Where X1's result goes.**
- `overall_idea.md`: step 3 (replace the "unverified" note with the verdict) and the key-numbers table (add the
  same-level rows).
- `2_models.ipynb`, Part 2.4: a **findings** cell after the code cell under *GNN with block premiums, and GNN +
  LightGBM blends* (cell 149 at the time of writing), with a **Decision** line, as in Parts 2.2 and 2.3. Also a run
  cell for the same-level arm before it.
- `2_models.ipynb`, Part 2.2 *E3 findings* (cell 56, "Blending buys nothing"): one sentence contrasting the boosters
  (errors correlate 0.99) with the tree + neural blend, using the correlation of the `lightgbm` and `neural` holdout
  errors from `pred` above.
- `3_evaluation.ipynb`, Part 3.3 *Summary* (cell 81): if the same-level biases no longer differ in sign, drop any
  wording that relies on it.

## X2. How many scored rows have no block history?

**Why.** Part 3.3 shows the GNN leans on neighbours' premiums only where a block (+ flat type) has no fit rentals.
That benefit can only matter for RMSE in proportion to how many test rows are in that situation.

In [ ]:
tr = cmp.clean_rent(pd.read_csv(cmp.DATA / "train.csv"))
te = cmp.clean_rent(pd.read_csv(cmp.DATA / "test.csv"))


def history_share(fit, scored):
    """% of scored rows by number of fit rentals in the same block, and block + flat type."""
    out = {}
    for name, keys in [("block", cmp.KEY), ("block + flat type", cmp.KEY + ["FLAT_TYPE"])]:
        n = scored.merge(fit.groupby(keys).size().rename("n").reset_index(), on=keys, how="left")["n"].fillna(0)
        out[name] = {"0": (n == 0).mean(), "1-2": n.between(1, 2).mean(), "3-5": n.between(3, 5).mean()}
    return pd.DataFrame(out).T.mul(100).round(2)


d = tr["RENT_APPROVAL_DATE"]
display(history_share(tr[d < cmp.HOLD_START], tr[(d >= cmp.HOLD_START) & (d < cmp.HOLD_END)]))
history_share(tr, te)

**Result (2026-10-30).** % of rows with no fit rentals of the same block + flat type: **1.44 holdout, 0.77 test**
(block alone: 0.56, 0.20). With 1 to 2 rentals: 7.17 holdout, 4.07 test. The holdout figure matches the 635 rows
in `by_history.csv`.

**Where it goes.** `overall_idea.md` step 2 (done); `2_models.ipynb` Part 2.4 findings after the code cell under
*Neighbours' premiums through the graph* (cell 151); `3_evaluation.ipynb` Part 3.3 *Summary* (cell 81), as the
reason the neighbour channel cannot move overall RMSE.